# 🪙 Entraînement de YOLOv8 sur Google Colab
### Projet Académique : Détection de Pièces Marocaines et Calcul du Montant Total

Ce notebook vous guidera à travers le processus complet :
1. Configuration de l'environnement GPU.
2. Installation d'**Ultralytics YOLOv8** et du SDK **Roboflow**.
3. Téléchargement automatique de vos images annotées depuis Roboflow.
4. Entraînement du modèle YOLOv8 sur GPU.
5. Analyse des performances (pertes, matrice de confusion, précision).
6. Téléchargement du fichier de poids optimal `best.pt` pour l'utiliser dans votre application Streamlit locale.

--- 
## Étape 1 : Activation et vérification du GPU

Pour entraîner un modèle de Deep Learning de manière efficace, l'utilisation d'un accélérateur graphique (GPU) est requise.
*   Allez dans **Exécution > Modifier le type d'exécution**.
*   Sélectionnez **T4 GPU** (ou supérieur) dans la section *Accélérateur matériel*.
*   Validez et exécutez la cellule suivante pour confirmer l'activation du GPU.

In [ ]:
!nvidia-smi

--- 
## Étape 2 : Installation des dépendances

Nous installons `ultralytics` pour le framework YOLOv8 et `roboflow` pour la gestion du dataset.

In [ ]:
%pip install ultralytics roboflow

In [ ]:
import os
import cv2
import matplotlib.pyplot as plt
from PIL import Image
from ultralytics import YOLO
from roboflow import Roboflow

--- 
## Étape 3 : Importation du Dataset depuis Roboflow

Après avoir annoté vos photos et généré une version de votre dataset sur Roboflow au format **YOLOv8** :
1. Cliquez sur **Export** dans Roboflow.
2. Sélectionnez **YOLOv8** et cochez **"Show Download Code"**.
3. Remplacez le code ci-dessous par la cellule générée par Roboflow (contenant votre clé API).

In [ ]:
# REMPLACEZ CE CODE PAR CELUI FOURNI PAR ROBOFLOW
rf = Roboflow(api_key="VOTRE_CLE_API_ROBOFLOW")
project = rf.workspace("votre-workspace").project("moroccan-coins-detection-xxxx")
dataset = project.version(1).download("yolov8")

### Visualisation du fichier `data.yaml` généré
Roboflow génère automatiquement un fichier `data.yaml` décrivant l'emplacement absolu des dossiers temporaires Colab et le nom des classes.

In [ ]:
# Afficher le fichier de configuration
!cat {dataset.location}/data.yaml

--- 
## Étape 4 : Entraînement de YOLOv8 sur le GPU

Nous chargeons un modèle pré-entraîné `yolov8n.pt` (modèle nano, très rapide et idéal pour un projet universitaire standard) et lançons l'entraînement avec les paramètres suivants :
*   `data` : Emplacement de notre fichier configuration yaml.
*   `epochs` : 50 époques (minimum pour commencer à obtenir de bons résultats).
*   `imgsz` : 640 (définition standard de YOLOv8).
*   `device` : 0 (pour forcer l'usage du GPU).

In [ ]:
# Charger le modèle YOLOv8 pré-entraîné sur COCO
model = YOLO('yolov8n.pt')

# Lancer l'entraînement sur notre dataset personnalisé
results = model.train(
    data=f"{dataset.location}/data.yaml",
    epochs=50,
    imgsz=640,
    batch=16,
    device=0,
    workers=2,
    name="yolov8_moroccan_coins"
)

--- 
## Étape 5 : Analyse des performances et des métriques

Une fois l'entraînement fini, YOLOv8 génère automatiquement des métriques d'évaluation dans le dossier `runs/detect/yolov8_moroccan_coins/`.

In [ ]:
# Afficher la courbe globale des métriques (Precision, Recall, mAP50, Losses)
results_img = "runs/detect/yolov8_moroccan_coins/results.png"
if os.path.exists(results_img):
    display(Image.open(results_img))
else:
    print("Courbe des résultats introuvable.")

In [ ]:
# Afficher la matrice de confusion pour analyser les erreurs de classification du modèle
confusion_matrix_img = "runs/detect/yolov8_moroccan_coins/confusion_matrix.png"
if os.path.exists(confusion_matrix_img):
    display(Image.open(confusion_matrix_img))
else:
    print("Matrice de confusion introuvable.")

--- 
## Étape 6 : Test de détection visuelle

Réalisons un test rapide de prédiction sur l'une des images du split test pour vérifier l'exactitude visuelle.

In [ ]:
# Charger le meilleur modèle entraîné
best_model = YOLO("runs/detect/yolov8_moroccan_coins/weights/best.pt")

# Prendre la première image du dossier test
test_dir = f"{dataset.location}/test/images"
if os.path.exists(test_dir) and len(os.listdir(test_dir)) > 0:
    test_img_name = os.listdir(test_dir)[0]
    test_img_path = os.path.join(test_dir, test_img_name)
    
    # Lancer l'inférence
    pred_results = best_model.predict(test_img_path, conf=0.5)
    
    # Récupérer et tracer l'image annotée
    plotted_img = pred_results[0].plot()
    plt.figure(figsize=(10, 10))
    plt.imshow(cv2.cvtColor(plotted_img, cv2.COLOR_BGR2RGB))
    plt.axis('off')
    plt.show()
else:
    print("Aucune image dans le dossier test à afficher.")

--- 
## Étape 7 : Téléchargement du fichier de poids optimal `best.pt`

Cette cellule déclenche l'invite de téléchargement du navigateur pour récupérer le fichier `best.pt`. 
Une fois téléchargé, déposez-le dans le dossier `models/` de votre structure locale de projet.

In [ ]:
from google.colab import files

weights_file = "runs/detect/yolov8_moroccan_coins/weights/best.pt"
if os.path.exists(weights_file):
    print("Préparation du téléchargement de best.pt...")
    files.download(weights_file)
else:
    print("Le fichier de poids best.pt n'existe pas. L'entraînement a-t-il été complété ?")